# 11 — Statistics, Residuals, Robustness, and Ablations

Purpose: Compute cluster-bootstrap intervals, Wilcoxon-Holm tests, residual diagnostics, sensor stress tests, and feature ablations.

Leakage rule: current-cycle full discharged capacity and SOH are labels only.
Preprocessing is fitted on training batteries, validation batteries control
selection and early stopping, and test batteries are scored exactly once.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v3"
RESULTS = ARTIFACT_ROOT / "results"
FIGURES = ARTIFACT_ROOT / "figures"
FEATURES = ARTIFACT_ROOT / "features"
for path in (RESULTS, FIGURES, FEATURES):
    path.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: E:\VIT\aiBatteryLifecycle\application


In [2]:
from scripts.run_statistical_analysis import run_statistical_analysis
outputs = run_statistical_analysis(PROJECT_ROOT, bootstrap_samples=10_000)
for name, table in outputs.items():
    display(name, table.head())

'averaged_predictions'

,model,row_index,battery_id,cycle_number,y_true,y_pred,residual,dataset
0,attention_lstm,0,B0005,0,100.550291,88.615417,11.934873,NASA
1,attention_lstm,1,B0005,1,100.000000,91.456011,8.543989,NASA
2,attention_lstm,2,B0005,2,99.405411,91.623639,7.781772,NASA
3,attention_lstm,3,B0005,3,99.400717,91.617999,7.782718,NASA
4,attention_lstm,4,B0005,4,99.367298,91.509270,7.858029,NASA


'residual_diagnostics'

,dataset,model,count,mean,std,median,min,max
0,Calce,attention_lstm,3789,0.344529,5.972401,0.044546,-37.119133,40.914627
1,Calce,battery_gpt,3789,0.701501,7.935017,0.544294,-33.326462,43.461241
2,Calce,bidirectional_lstm,3789,0.568204,9.447270,0.250623,-50.016126,57.116036
3,Calce,dynamic_graph_itransformer,3789,-0.362929,6.738676,-0.274275,-42.286461,42.016379
4,Calce,extra_trees,3789,0.181964,1.949759,0.160070,-14.099204,12.252524


'nasa_per_battery_metrics'

,mae,mse,rmse,r2,adjusted_r2,mape,median_absolute_error,p95_absolute_error,bias,within_5pp,battery_id,n_samples,model
0,8.083091,83.091965,9.115479,0.214082,NaN,10.116053,7.770602,14.524479,4.436559,0.250000,B0005,168,attention_lstm
1,17.480770,447.666829,21.158139,-1.907542,NaN,25.782319,21.120829,35.341529,13.819185,0.226190,B0006,168,attention_lstm
2,5.798971,41.492923,6.441500,0.430092,NaN,6.751076,6.023616,10.686056,2.155120,0.392857,B0007,168,attention_lstm
3,6.435839,55.277520,7.434885,0.211181,NaN,7.800145,6.885523,11.559971,1.659893,0.371212,B0018,132,attention_lstm
4,6.420816,45.246150,6.726526,-31.220564,NaN,6.510028,6.521500,8.627115,-6.420816,0.107143,B0025,56,attention_lstm


... (6 outputs left out)

'feature_ablation'

,fold,ablation,n_features,mae,mse,rmse,r2,adjusted_r2,mape,median_absolute_error,p95_absolute_error,bias,within_5pp
0,1,all_safe_features,18,2.146186,9.878200,3.142960,0.847561,0.838322,2.511267,1.226565,6.871231,0.516168,0.867089
1,1,without_temperature,12,2.480713,10.701084,3.271251,0.834862,0.828322,2.880331,1.824778,6.682496,0.327485,0.886076
2,1,without_voltage,12,2.246777,9.751563,3.122749,0.849515,0.843556,2.626636,1.351036,6.678455,0.691128,0.873418
3,1,without_current,16,2.110333,9.837342,3.136454,0.848192,0.840068,2.450507,1.203120,6.877560,0.196531,0.860759
4,1,time_and_usage_only,3,2.624228,11.950166,3.456901,0.815587,0.813814,3.007995,2.088944,7.048400,1.298473,0.841772


'sensor_stress'

,fold,scenario,mae,mse,rmse,r2,adjusted_r2,mape,median_absolute_error,p95_absolute_error,bias,within_5pp
0,1,unmodified,2.146186,9.878200,3.142960,0.847561,0.838322,2.511267,1.226565,6.871231,0.516168,0.867089
1,1,temperature_missing,4.582617,35.979322,5.998277,0.444773,0.411122,5.442797,2.820797,11.796101,3.591457,0.613924
2,1,voltage_current_noise_5pct_sd,2.195137,10.144470,3.185038,0.843452,0.833964,2.557402,1.282113,6.861404,0.348841,0.876582
3,2,unmodified,2.215180,10.735863,3.276563,0.887605,0.882922,2.667516,1.536275,5.316511,1.379167,0.922395
4,2,temperature_missing,3.891151,31.233011,5.588650,0.673019,0.659394,4.721919,2.127190,12.317898,2.583876,0.705100
